# CRL focus explorer

Run all once, then change parameters and press **Compute focus**. Source sizes are **intensity FWHM**; detector z is measured from the CRL exit.


In [ ]:
#@title Prepare XPOSim and interactive controls
import hashlib
import importlib.util
import re
import subprocess
import sys
import urllib.request
from pathlib import Path

SOURCE_REF = '9ae53a42f0f8beec403355e078821689fa6ff43a'
SOURCE_SHA256 = '592bc177ba8e6be5852c9a370ba80b7ab5350760d46bfd6f52dee9a6f1f96e57'
PROJECT_DIR = Path.cwd() / '.xposim_runtime' / SOURCE_REF
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
requirements = {'numpy':'numpy', 'scipy':'scipy', 'matplotlib':'matplotlib', 'ipywidgets':'ipywidgets>=8,<9'}
missing = [requirement for name,requirement in requirements.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',*missing])
try:
    gpu_info = subprocess.run(['nvidia-smi'],capture_output=True,text=True,timeout=15)
except (FileNotFoundError,subprocess.TimeoutExpired):
    gpu_info = None
if gpu_info is not None and gpu_info.returncode==0 and importlib.util.find_spec('cupy') is None:
    try:
        toolkit = subprocess.run(['nvcc','--version'],capture_output=True,text=True,timeout=15)
        release = re.search(r'release\s+(\d+)\.',toolkit.stdout)
    except (FileNotFoundError,subprocess.TimeoutExpired):
        release = None
    if release is None:
        release = re.search(r'CUDA Version:\s*(\d+)\.',gpu_info.stdout)
    major = int(release.group(1)) if release else None
    wheel = {12:'cupy-cuda12x',13:'cupy-cuda13x'}.get(major)
    if wheel:
        try:
            subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',f'{wheel}==14.2.0'])
            importlib.invalidate_caches()
        except subprocess.CalledProcessError:
            print('CuPy installation failed. Auto backend will use NumPy when CUDA is unavailable.')
    else:
        print('CUDA version was not recognized. CPU execution remains available.')
for relative in ['XPOSim.py','xray_material_data/AtomicConstants.dat',
                 'xray_material_data/CrossSec-Compton_McMaster.dat','xray_material_data/f1f2_Windt.dat']:
    target = PROJECT_DIR / relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists():
        url = f'https://raw.githubusercontent.com/mamichberdey/XPOSim/{SOURCE_REF}/{relative}'
        with urllib.request.urlopen(url,timeout=90) as response:
            content = response.read()
        temporary = target.with_suffix(target.suffix+'.download')
        temporary.write_bytes(content)
        temporary.replace(target)
if hashlib.sha256((PROJECT_DIR/'XPOSim.py').read_bytes()).hexdigest()!=SOURCE_SHA256:
    raise RuntimeError('XPOSim checksum mismatch. Remove .xposim_runtime and repeat preparation.')
spec = importlib.util.spec_from_file_location('XPOSim',PROJECT_DIR/'XPOSim.py')
xpos = importlib.util.module_from_spec(spec)
sys.modules['XPOSim'] = xpos
spec.loader.exec_module(xpos)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
print(f'XPOSim {xpos.__version__}; available backend: {xpos.select_backend("auto")}.')
print('Ready. Change parameters below and press Compute focus.')


In [ ]:
#@title CRL, source and numerical controls
import html
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
try:
    from google.colab import output as colab_output
except ImportError:
    colab_output = None
if colab_output is not None:
    colab_output.enable_custom_widget_manager()

controls = {}
containers = {}
def input_field(name, label, widget, tooltip=''):
    controls[name] = widget
    widget.layout.width = '100%'
    label_html = widgets.HTML('<span title="' + html.escape(tooltip, quote=True) + '">' + html.escape(label) + '</span>')
    box = widgets.VBox([label_html, widget], layout=widgets.Layout(width='100%'))
    containers[name] = box
    return box

def group(fields):
    return widgets.GridBox(fields, layout=widgets.Layout(
        grid_template_columns='repeat(auto-fit, minmax(190px, 1fr))', grid_gap='8px 14px', width='100%'))

optics = group([
    input_field('focusing', 'CRL focusing', widgets.Dropdown(options=['One-dimensional focusing', 'Two-dimensional focusing'])),
    input_field('grid', 'Calculation grid', widgets.Dropdown(options=['1D', '2D'])),
    input_field('backend', 'Compute backend', widgets.Dropdown(options=['Auto', 'CuPy / GPU', 'NumPy / CPU'])),
    input_field('accuracy', 'Sampling', widgets.Dropdown(options=['Quick', 'Refined'])),
    input_field('material', 'Material', widgets.Dropdown(options=['Beryllium (Be)', 'Silicon (Si)', 'Aluminium (Al)', 'Diamond (C)', 'Custom'])),
    input_field('formula', 'Chemical formula', widgets.Text(value='SiO2')),
    input_field('density', 'Density (g/cm³)', widgets.FloatText(value=2.2)),
    input_field('energy', 'Photon energy (keV)', widgets.FloatText(value=12.0)),
    input_field('radius', 'Surface radius (µm)', widgets.FloatText(value=50.0)),
    input_field('aperture', 'Full aperture (µm)', widgets.FloatText(value=100.0)),
    input_field('bridge', 'Bridge thickness (µm)', widgets.FloatText(value=30.0)),
    input_field('elements', 'CRL elements', widgets.BoundedIntText(value=10, min=1, max=10000)),
    input_field('depth', 'Structure depth along y (µm)', widgets.FloatText(value=50.0)),
])
source_controls = group([
    input_field('source', 'Source', widgets.Dropdown(options=['Point source', 'Gaussian source (incoherent)'])),
    input_field('distance', 'Source-to-CRL distance (m)', widgets.FloatText(value=15.0)),
    input_field('fwhm_x', 'Source FWHM x (µm)', widgets.FloatText(value=5.0)),
    input_field('fwhm_y', 'Source FWHM y (µm)', widgets.FloatText(value=5.0)),
    input_field('source_method', 'Extended-source method', widgets.Dropdown(options=['Gaussian convolution', 'Incoherent point sampling'])),
    input_field('source_points', 'Source nodes per axis', widgets.BoundedIntText(value=7, min=3, max=101, step=2), 'Gaussian quadrature order. Increase it to check convergence.'),
])
advanced_fields = group([
    input_field('adaptive_window', 'Adapt propagation window', widgets.Checkbox(value=True, indent=False)),
    input_field('nx', 'Grid points x (0 = auto)', widgets.IntText(value=0)),
    input_field('ny', 'Grid points y (0 = auto)', widgets.IntText(value=0)),
    input_field('window', 'Initial window / geometry factor', widgets.FloatText(value=1.3)),
    input_field('batch', 'Batch size (0 = auto)', widgets.IntText(value=0)),
    input_field('z_mode', 'Displayed z ROI', widgets.Dropdown(options=['Auto around numerical focus', 'Manual bounds'])),
    input_field('z_min', 'z minimum from CRL exit (mm)', widgets.FloatText(value=1000.0)),
    input_field('z_max', 'z maximum from CRL exit (mm)', widgets.FloatText(value=1250.0)),
    input_field('z_points', 'Displayed z samples', widgets.BoundedIntText(value=51, min=5, max=501)),
    input_field('z_level', 'Axial ROI intensity fraction', widgets.BoundedFloatText(value=0.2, min=0.001, max=0.99, step=0.05)),
    input_field('xy_level', 'Transverse ROI intensity fraction', widgets.BoundedFloatText(value=0.01, min=0.00001, max=0.5, step=0.01)),
    input_field('tolerance', 'Focus z tolerance (µm; 0 = auto)', widgets.FloatText(value=0.0)),
    input_field('display_floor', 'Colormap intensity floor', widgets.BoundedFloatText(value=0.0001, min=1e-10, max=0.5)),
])
advanced = widgets.Accordion(children=[advanced_fields], selected_index=None)
advanced.set_title(0, 'Sampling, ROI and search')

def sync_controls(change=None):
    two_focusing = controls['focusing'].value == 'Two-dimensional focusing'
    if two_focusing:
        controls['grid'].value = '2D'
    controls['grid'].disabled = two_focusing
    two_grid = controls['grid'].value == '2D'
    gaussian = controls['source'].value != 'Point source'
    visibility = {
        'depth': two_grid and not two_focusing,
        'formula': controls['material'].value == 'Custom',
        'density': controls['material'].value == 'Custom',
        'fwhm_x': gaussian, 'fwhm_y': gaussian and two_grid,
        'source_method': gaussian,
        'source_points': gaussian and controls['source_method'].value == 'Incoherent point sampling',
        'ny': two_grid,
        'z_min': controls['z_mode'].value == 'Manual bounds',
        'z_max': controls['z_mode'].value == 'Manual bounds',
    }
    for name, visible in visibility.items():
        containers[name].layout.display = '' if visible else 'none'
for name in ['focusing', 'grid', 'source', 'source_method', 'material', 'z_mode']:
    controls[name].observe(sync_controls, names='value')
sync_controls()

def read_configuration():
    v = {name: widget.value for name, widget in controls.items()}
    for name in (('nx', 'batch', 'tolerance', 'ny') if v['grid']=='2D' else ('nx', 'batch', 'tolerance')):
        if v[name] < 0:
            raise ValueError(f'{name} must be non-negative; zero selects automatic control.')
    presets = {'Beryllium (Be)': ('Be', 1.848), 'Silicon (Si)': ('Si', 2.33),
               'Aluminium (Al)': ('Al', 2.70), 'Diamond (C)': ('C', 3.515)}
    formula, density = presets.get(v['material'], (v['formula'], v['density']))
    return xpos.CRLFocusConfig(
        energy_kev=v['energy'], material_formula=formula, density_g_cm3=density,
        radius_m=v['radius']*1e-6, aperture_m=v['aperture']*1e-6,
        bridge_thickness_m=v['bridge']*1e-6, n_elements=v['elements'],
        geometry='two-dimensional' if v['focusing']=='Two-dimensional focusing' else 'one-dimensional',
        grid_dimension=2 if v['grid']=='2D' else 1,
        structure_depth_m=v['depth']*1e-6,
        source_distance_m=v['distance'],
        source_type='point' if v['source']=='Point source' else 'gaussian',
        source_fwhm_x_m=v['fwhm_x']*1e-6 if v['source']!='Point source' else 0.0,
        source_fwhm_y_m=v['fwhm_y']*1e-6 if v['source']!='Point source' and v['grid']=='2D' else 0.0,
        source_method='points' if v['source_method']=='Incoherent point sampling' else 'convolution',
        source_points=v['source_points'],
        backend={'Auto':'auto','CuPy / GPU':'cupy','NumPy / CPU':'numpy'}[v['backend']],
        accuracy='preview' if v['accuracy']=='Quick' else 'refined',
        grid_points=v['nx'] or None, grid_points_y=(v['ny'] or None) if v['grid']=='2D' else None,
        window_factor=v['window'], adaptive_window=v['adaptive_window'], batch_size=v['batch'] or None,
        scan_points=v['z_points'],
        scan_range_m=None if v['z_mode']=='Auto around numerical focus' else (v['z_min']*1e-3,v['z_max']*1e-3),
        axial_roi_level=v['z_level'], transverse_roi_level=v['xy_level'],
        focus_tolerance_m=v['tolerance']*1e-6 if v['tolerance'] else None,
    )

display(widgets.VBox([widgets.HTML('<b>CRL</b>'), optics,
    widgets.HTML('<b>Source</b>'), source_controls, advanced]))


In [ ]:
#@title Focus analysis and interactive detector plane
import gc
import time
from matplotlib.colors import LogNorm

CMAP = plt.get_cmap('CMRmap')
state = {'result': None, 'slider': None, 'handler': None, 'plane_updates': 0, 'error': None}
scan_output = widgets.Output()
plane_output = widgets.Output()
plane_controls = widgets.VBox()
compute_button = widgets.Button(description='Compute focus', button_style='primary', icon='play')

def number(value, scale=1.0, precision=5):
    return 'unresolved' if value is None or not np.isfinite(value) else f'{value*scale:.{precision}g}'

def table(rows):
    display(HTML('<table>' + ''.join('<tr><td>'+html.escape(str(a))+'</td><td><b>'+html.escape(str(b))+'</b></td></tr>' for a,b in rows) + '</table>'))

def plot_scan(result):
    cfg = result.configuration
    magnification = ' / '.join(number(m) for m in result.magnification_at_focus)
    table([
        ('Analytical BFL, collimated (mm)', number(result.collimated_focus_m,1e3)),
        ('Analytical image distance (mm)', number(result.analytical_focus_m,1e3)),
        ('Numerical local focus (mm)', number(result.peak_z_m,1e3)),
        ('Magnification at numerical focus, x / y', magnification),
        ('CRL length (mm) / transmission', number(result.lens_length_m,1e3)+' / '+f'{result.transmission:.2%}'),
        ('Window x / y (µm)', ' / '.join(number(v,1e6) for v in ([result.grid_step_m*result.grid_shape[-1]] if cfg.grid_dimension==1 else [result.grid_step_m*result.grid_shape[-1],result.grid_step_y_m*result.grid_shape[0]]))),
        ('Backend / full grid / focus evaluations', f'{result.backend} / {result.grid_shape} / {result.numerical_focus.evaluations}'),
    ])
    for message in result.diagnostics:
        prefix, _, text = message.partition(':')
        display(HTML(('<small>' if prefix in {'convolution_model','mesh_window'} else '<div><b>Check:</b> ')
                     + html.escape(text.strip()) + ('</small>' if prefix in {'convolution_model','mesh_window'} else '</div>')))
    z_mm = result.z_m*1e3
    scale = max(float(np.max(result.longitudinal_intensity)),np.finfo(float).tiny)
    figure = plt.figure(figsize=(10,8),constrained_layout=True)
    spec = figure.add_gridspec(3,2,width_ratios=[1,0.025],height_ratios=[3,1,1])
    map_axis = figure.add_subplot(spec[0,0])
    peak_axis = figure.add_subplot(spec[1,0],sharex=map_axis)
    width_axis = figure.add_subplot(spec[2,0],sharex=map_axis)
    color_axis = figure.add_subplot(spec[0,1])
    mesh = map_axis.pcolormesh(z_mm,result.x_m*1e6,result.longitudinal_intensity.T/scale,
        shading='auto',cmap=CMAP,norm=LogNorm(vmin=controls['display_floor'].value,vmax=1))
    figure.colorbar(mesh,cax=color_axis,label='I / map Imax')
    map_axis.set(ylabel='x (µm)',title='x–z intensity through y ≈ 0')
    peak_axis.plot(z_mm,result.peak_intensity)
    peak_axis.set(ylabel='Plane peak intensity\n(relative)')
    if result.fwhm_m.ndim==1:
        width_axis.plot(z_mm,result.fwhm_m*1e6,label='FWHM x')
    else:
        width_axis.plot(z_mm,result.fwhm_m[:,0]*1e6,label='FWHM x')
        if cfg.geometry=='two-dimensional':
            width_axis.plot(z_mm,result.fwhm_m[:,1]*1e6,'--',label='FWHM y')
    width_axis.set(xlabel='z from CRL exit (mm)',ylabel='FWHM (µm)')
    width_axis.legend()
    for ax in [map_axis,peak_axis,width_axis]:
        ax.axvline(result.peak_z_m*1e3,color='gray',ls=':',lw=1)
    for ax in [peak_axis,width_axis]:
        ax.grid(alpha=.2)
    for ax in [map_axis,peak_axis]:
        ax.tick_params(labelbottom=False)
    plt.show()
    plt.close(figure)


def draw_plane(z_mm):
    result = state['result']
    if result is None:
        return
    plane = result.plane_at(float(z_mm)*1e-3)
    state['plane_updates'] += 1
    state['last_plane'] = plane
    with plane_output:
        clear_output(wait=True)
        widths = np.atleast_1d(plane.metrics.fwhm)
        table([
            ('z from CRL exit (mm)',number(plane.z_m,1e3)),
            ('Peak intensity / FWHM x / y (µm)',number(plane.metrics.max_intensity)+' / '+' / '.join(number(w,1e6) for w in widths)),
            ('Source magnification x / y', ' / '.join(number(m) for m in plane.magnification)),
        ])
        scale = max(float(plane.metrics.max_intensity),np.finfo(float).tiny)
        if plane.y_m is None:
            figure,ax = plt.subplots(figsize=(9,3))
            ax.plot(plane.x_m*1e6,plane.intensity/scale)
            ax.set(xlabel='x (µm)',ylabel='I / plane Imax',title='Transverse intensity')
            ax.grid(alpha=.2)
        else:
            figure,axes = plt.subplots(1,3,figsize=(13,3.5),constrained_layout=True)
            mesh=axes[0].pcolormesh(plane.x_m*1e6,plane.y_m*1e6,plane.intensity/scale,
                shading='auto',cmap=CMAP,norm=LogNorm(vmin=controls['display_floor'].value,vmax=1))
            axes[0].set(xlabel='x (µm)',ylabel='y (µm)',title='Transverse intensity')
            figure.colorbar(mesh,ax=axes[0],label='I / plane Imax')
            j,i=np.unravel_index(np.argmax(plane.intensity),plane.intensity.shape)
            axes[1].plot(plane.x_m*1e6,plane.intensity[j,:]/scale)
            axes[1].set(xlabel='x (µm)',ylabel='I / plane Imax',title='Slice through the peak')
            axes[2].plot(plane.y_m*1e6,plane.intensity[:,i]/scale)
            axes[2].set(xlabel='y (µm)',ylabel='I / plane Imax',title='Slice through the peak')
        plt.show()
        plt.close(figure)


def compute_focus(_=None):
    compute_button.disabled=True
    state['error']=None
    if state['slider'] is not None:
        state['slider'].unobserve(state['handler'],names='value')
        state['slider'].close()
    state['slider']=None
    state['handler']=None
    state['result']=None
    state.pop('last_plane',None)
    plane_controls.children=()
    plane_output.clear_output()
    gc.collect()
    try:
        config=read_configuration()
        with scan_output:
            clear_output(wait=True)
            started=time.perf_counter()
            result=xpos.estimate_crl_focus(config)
            state['result']=result
            plot_scan(result)
            print(f'Elapsed: {time.perf_counter()-started:.2f} s. Focus search is independent of displayed z sampling.')
        low,high=float(result.z_m[0]*1e3),float(result.z_m[-1]*1e3)
        slider=widgets.FloatSlider(value=result.peak_z_m*1e3,min=low,max=high,
            step=max((high-low)/400,1e-8),readout_format='.5f',description='z (mm)',
            continuous_update=False,style={'description_width':'initial'},layout=widgets.Layout(width='95%'))
        def changed(change):
            draw_plane(change['new'])
        slider.observe(changed,names='value')
        state['slider'],state['handler']=slider,changed
        plane_controls.children=(slider,)
        draw_plane(slider.value)
    except Exception as exc:
        state['error']=exc
        state['result']=None
        with scan_output:
            clear_output(wait=True)
            display(HTML('<b>Calculation stopped:</b> '+html.escape(str(exc))))
    finally:
        compute_button.disabled=False

compute_button.on_click(compute_focus)
display(compute_button,scan_output)
display(HTML('<b>Interactive transverse plane</b>'),plane_controls,plane_output)
compute_focus()


<details><summary>Model and numerical notes</summary>

- One-dimensional focusing can use either a 1D grid or a 2D grid with a finite structure depth along y. Two-dimensional focusing requires a 2D grid.
- Gaussian sources are incoherent. Point sampling propagates weighted Gaussian quadrature modes; convolution is a faster shift-invariant intensity approximation with a z-dependent source-to-plane scale.
- Focus is a local peak found by adaptive bracketing and bounded scalar optimization. Display z samples are created afterwards, within an intensity-based ROI.
- The initial window fits the CRL aperture/depth with margin. Sampling resolves phase and the expected focal width. Edge probes can expand the window; Quick/Refined is a resolution choice, not a convergence certificate.
- Compare finer grids, larger windows and more source nodes when quantitative accuracy matters. Intensities are relative; no calculations are saved or exported.
- [Usage and model guide](https://github.com/mamichberdey/XPOSim/blob/codex/xposim-correctness-gpu-batching/COLAB_GUIDE.md)

</details>
